This project is for daily and hourly weather data analysis to:
- determine prevailing winter wind direction which is one of key factor for snow fence design.
- generate snow report for helping civic engineer to select snow year data for their snow flux model or snowdrift model and snow mitigration.

The study area is proposed highway 413, a proposed highway and transit corridor running through York, Peel and Halton Regions.
https://highway413.ca/en/ or [Highway 413 ArcGIS online](https://www.arcgis.com/apps/mapviewer/index.html?url=https://services1.arcgis.com/9NvE8jKNWWlDGsUJ/ArcGIS/rest/services/Highway_413_River_Crossings_WFL1/FeatureServer/2&source=sdhttps://)

This capstone project is to answer folling questions
- Q1: How to perform winter weather analysis to determine prevailing winter wind direction in the study area?
- Q2: How to perform winter weather analysis to generate the snow report for helping civic engineers to set snow flux models or snowdrift modelling


Data source is from Environment and Climate Change Canada weather stations (environ thousands weather stations)
https://climate.weather.gc.ca/historical_data/search_historic_data_e.html

Station selection Criteria:
- close to study area,
- availability of daily and hourly data
- have enough long observation history,
- have data of key attributes for snow and winter wind analysis

[Stations within 50km of study area](https://climate.weather.gc.ca/historical_data/search_historic_data_stations_e.html?searchType=stnProx&timeframe=1&txtRadius=50&optProxType=city&selCity=43%7C35%7C79%7C37%7CMississauga&selPark=&txtCentralLatDeg=&txtCentralLatMin=&txtCentralLatSec=&txtCentralLongDeg=&txtCentralLongMin=&txtCentralLongSec=&txtLatDecDeg=&txtLongDecDeg=&StartYear=1840&EndYear=2026&optLimit=specDate&Year=2026&Month=1&Day=10&selRowPerPage=25)

==> Station Toronto Intl A (Station ID = 51459) is selected with have daily and hourly weather data from 2013 to date.


Data Collection

Use API with base http://climate.weather.gc.ca/climate_data/bulk_data_e.html and parameters
API_params = (
  'format=csv'
  &'stationID={stationID}’
  &Year={start year}
  &Month={start month}
  &Day= {start day)
  &timeframe={2 for daily, 1 for hourly)
  &submit=Download+Data
)

Below is a program can be used to download historiscal weather data, run in Python.

In [31]:
'''
import os
import requests
import time

# Variables should be changed before performing the process
stationIDs = ['51459']    #can be used to download data from multiple stations
startYear = 2013
endYear = 2025
dailyFolder = 'Daily'
hourlyFolder = 'Hourly'

# create data folders
if not os.path.exists(dailyFolder):
    os.makedirs(dailyFolder)

if not os.path.exists(hourlyFolder):
    os.makedirs(hourlyFolder)

# download daily data
for x in stationIDs:
    station = x.strip()
    for year in range(startYear, endYear+1):
        #1. Download each month of hourly weather CSV file for the station
        print('\n- Daily data - Station {0} for {1}...'.format(station, year))
        url = 'http://climate.weather.gc.ca/climate_data/bulk_data_e.html?format=csv&stationID={0}&Year={1}&Month=1&Day=1&timeframe=2&submit=Download+Data'.format(station, year)
        csvFileName = 'Station_{0}_{1}.csv'.format(station, year)
        response = requests.get(url)
        with open(os.path.join(dailyFolder, csvFileName), 'wb') as f:
            f.write(response.content)

# download hourly data
for x in stationIDs:
    station = x.strip()
    for year in range(startYear, endYear+1):
        for month in range(1, 12+1):

            #1. Download each month of hourly weather CSV file for the station
            print('\n- Hourly data - Station {0} for {1}-{2}...'.format(station, year, month))
            url = 'http://climate.weather.gc.ca/climate_data/bulk_data_e.html?format=csv&stationID={0}&Year={1}&Month={2}&Day=1&timeframe=1&submit=Download+Data'.format(station, year, month )
            csvFileName = 'Station_{0}_{1}_{2}.csv'.format(station, year, month)
            response = requests.get(url)
            with open(os.path.join(hourlyFolder, csvFileName), 'wb') as f:
                f.write(response.content)
'''

''' Check availability of key variables for wind and snow analysis '''
stationID = '51459'
startYear = 2013
endYear = 2025

print('\nHourly ... ')
# Hourly
hdf = pd.DataFrame()
for year in range(startYear, endYear+1):
  for month in range(1, 13):
    hourly_df = pd.read_csv(f'Station_{stationID}_{year}_{month}.csv', dtype={'Date/Time': str})
    hdf = pd.concat([hdf, hourly_df], ignore_index=True)

hdf.info()

print('\nDaily ...')
# Daily
ddf = pd.DataFrame()
for year in range(startYear, endYear):
    daily_df = pd.read_csv(f'Station_{stationID}_{year}.csv', dtype={'Date/Time': str})
    ddf = pd.concat([ddf, daily_df], ignore_index=True)

ddf.info()



Hourly ... 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113952 entries, 0 to 113951
Data columns (total 31 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Longitude (x)        113952 non-null  float64
 1   Latitude (y)         113952 non-null  float64
 2   Station Name         113952 non-null  object 
 3   Climate ID           113952 non-null  int64  
 4   Date/Time (LST)      113952 non-null  object 
 5   Year                 113952 non-null  int64  
 6   Month                113952 non-null  int64  
 7   Day                  113952 non-null  int64  
 8   Time (LST)           113952 non-null  object 
 9   Flag                 0 non-null       float64
 10  Temp (°C)            110040 non-null  float64
 11  Temp Flag            2 non-null       object 
 12  Dew Point Temp (°C)  110033 non-null  float64
 13  Dew Point Temp Flag  10 non-null      object 
 14  Rel Hum (%)          110028 non-null  float64
 15  Rel 

Daily weather data analysis to generate a report on snow data.
2 types of snow data:
- Total Snow (cm): is attribute for daily snow fall
- Snow on Grnd (cm): is attribute for daily snow on ground.


In [32]:
import pandas as pd
import numpy as np

''' init variables * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * '''
stationID = '51459'
sYear = 2013
eYear   = 2025

''' Step 1. read daily csv files to data frame * * * * * * * * * * * * * * * * * * * * * * * * '''
cols = ['Date/Time', 'Year', 'Month', 'Day', 'Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)',
        'Total Rain (mm)', 'Total Snow (cm)', 'Total Precip (mm)', 'Snow on Grnd (cm)' ]
df = pd.DataFrame()
for year in range(sYear, eYear+1):
    daily_df = pd.read_csv(f'Station_{stationID}_{year}.csv', usecols=cols, dtype={'Date/Time': str})
    df = pd.concat([df, daily_df], ignore_index=True)


''' Step 2. Cleaning, Preparing * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *'''
'''
Create snow_year column
the Snow Year (SY) using the starting period from November 1st to April 30th, which is then refined for specific
dates based on presence of snow on the ground. The calendar period used for SY, is defined as the calendar year in
which the corresponding winter season ends; for example, the 2014/2015 winter season is referred to as the SY2015.
'''

# create a list of snow year conditions
snow_year_conditions = [
  (df['Month'] <= 4),
  (df['Month'] > 5) & (df['Month'] <= 10),
  (df['Month'] > 10)
]

# create a list of the values to assign for each snow year condition
snow_year_values = [df['Year'], '', df['Year']+1]

# create a new column snow year and use np.select to assign values to it using our lists as arguments
df['snow_year'] = 'SY' + np.select(snow_year_conditions, snow_year_values)


''' Replace the NaN with 0 for the column 'Snow on Grnd (cm)' '''
df[['Snow on Grnd (cm)']] = df[['Snow on Grnd (cm)']].fillna(value=0)

''' Drop rows if 'Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)',
    'Total Rain (mm)', 'Total Snow (cm)', 'Total Precip (mm)' are empty '''
df.dropna(axis = 0, subset = ['Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)', 'Total Rain (mm)', 'Total Snow (cm)', 'Total Precip (mm)'], how = 'all', inplace = True)

''' fill in missing data values in the total snow
If both rainfall and snowfall depths are missing, the total precipitation is split into the liquid (rain)
and solid (snow) components based on air temperature sing the following logic:
- If the maximum daily temperature is 0°C or below, all precipitation is allocated to the
snow component
- If the minimum daily temperature is above 0°C, all precipitation is allocated to the rainfall
component
- In all other cases, the decision is based on the mean daily temperature
    + Precipitation is assumed to be snow if the mean daily temperature is below -1°C,
and rain if the mean daily temperature is above +1°C
    + If the mean daily temperature is between -1 and +1°C, it is split evenly into snow
and rain
'''
def conditions(row):
  val = 0
  if (pd.notna(row['Total Snow (cm)'])):
    val = row['Total Snow (cm)']
  elif (pd.isnull(row['Total Rain (mm)']) & pd.isnull(row['Total Snow (cm)']) & pd.notna(row['Total Precip (mm)'])):
    if (row['Max Temp (°C)'] < 0) | (row['Mean Temp (°C)'] < -1):
      val = row['Total Precip (mm)']
    elif (row['Mean Temp (°C)'] >= -1) & (row['Mean Temp (°C)'] < 1):
      val = row['Total Precip (mm)']/2
  else:
    val = 0
  return val

df['Total Snow (cm)']= df.apply(conditions, axis=1)


# remove SY2026 which has not enough data from November 1st 2025 to April 30th 2026
df = df[df['snow_year'] != 'SY2026']

# add formatted date time (in format MMM DD) column for x-axis chart
df['formatted_datetime'] = pd.to_datetime(df['Date/Time']).dt.strftime('%b %d')

df.info()

# Filter rows where 'month' is in period from November 1st to April 30th
snow_months = [1, 2, 3, 4, 11, 12]
snow_season_df = df[df['Month'].isin(snow_months)]

# snow_season_df group by snow_year
snow_season_groups = snow_season_df.groupby('snow_year')

# snow season list
snow_years = list(snow_season_groups.groups.keys())


print(snow_season_df[['Date/Time', 'formatted_datetime', 'Total Rain (mm)', 'Total Snow (cm)', 'Total Precip (mm)', 'Snow on Grnd (cm)', 'snow_year']].head())
print(snow_years)



<class 'pandas.core.frame.DataFrame'>
Index: 4514 entries, 163 to 4686
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date/Time           4514 non-null   object 
 1   Year                4514 non-null   int64  
 2   Month               4514 non-null   int64  
 3   Day                 4514 non-null   int64  
 4   Max Temp (°C)       4498 non-null   float64
 5   Min Temp (°C)       4497 non-null   float64
 6   Mean Temp (°C)      4495 non-null   float64
 7   Total Rain (mm)     4486 non-null   float64
 8   Total Snow (cm)     4514 non-null   float64
 9   Total Precip (mm)   4498 non-null   float64
 10  Snow on Grnd (cm)   4514 non-null   float64
 11  snow_year           4514 non-null   object 
 12  formatted_datetime  4514 non-null   object 
dtypes: float64(7), int64(3), object(3)
memory usage: 493.7+ KB
      Date/Time formatted_datetime  Total Rain (mm)  Total Snow (cm)  \
304  2013-11-01          

Snow on ground (attribute 'Snow on grnd (cm)') analysis

Generate snow season based statistic (sum, max, mean, count) report and charts

In [33]:
''' Create Snow Accumulation Season charts '''
import math
from bokeh.plotting import figure, output_notebook, show, save
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.layouts import gridplot
from bokeh.plotting import output_file
output_notebook()


''' Snow on the ground analysis * * * * * * * * * * * * * * * * * * * * * * * * * * '''
stat1_col = 'Snow on Grnd (cm)'
# Figure 1: SAS - Snow on the ground Accumulation Season (SUM of Snow on Grnd (cm))
sas1 = snow_season_groups.agg(sum=(stat1_col, 'sum'))
sas1_source = ColumnDataSource(data = dict(snow_year = sas1.index.values, sum = sas1['sum']))
sas1_fig = figure(title = 'Snow on Ground Accumulation (SUM)', x_axis_label = 'Snow Year',
                  y_axis_label = 'Snow Depth (cm)', height=350, x_range = snow_years, toolbar_location=None)
sas1_fig.vbar(x='snow_year', top='sum', width=0.5, color= 'DarkCyan', source = sas1_source)
sas1_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @sum (cm)</font>'))
sas1_fig.xgrid.grid_line_color = None
sas1_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels

print(sas1)

''' Snow on the ground analysis * * * * * * * * * * * * * * * * * * * * * * * * * * '''
# Figure 2: SMS - Daily Maximum Snow on Ground of Season (MAX  of Snow on Grnd (cm))
sms1 = snow_season_groups.agg(max=(stat1_col, 'max'))
sms1_source = ColumnDataSource(data = dict(snow_year = sms1.index.values, max = sms1['max']))
sms1_fig = figure(title = 'Daily Maximum Snow on Ground (MAX)', x_axis_label = 'Snow Year',
                  y_axis_label = 'Snow Depth (cm)', height=350, x_range = snow_years, toolbar_location=None)
sms1_fig.vbar(x='snow_year', top='max', width=0.5, color= 'Tomato', source = sms1_source)
sms1_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @max (cm)</font>'))
sms1_fig.xgrid.grid_line_color = None
sms1_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels


# Figure 3: SES - Daily mean/average/norm snow of Season (MEAN of Snow on Grnd (cm))
sns1 = snow_season_groups.agg(mean=(stat1_col, 'mean'))
sns1_source = ColumnDataSource(data = dict(snow_year = sns1.index.values, mean = sns1['mean']))
sns1_fig = figure(title = 'Daily Mean Snow on Ground (MEAN)', x_axis_label = 'Snow Year',
                  y_axis_label = 'Snow Depth (cm)', height=350, x_range = snow_years, toolbar_location=None)
sns1_fig.vbar(x='snow_year', top='mean', width=0.5, color= 'Olive', source = sns1_source)
sns1_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @mean (cm)</font>'))
sns1_fig.xgrid.grid_line_color = None
sns1_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels

# Figure 4: SCS - Count Daily snow of Season (COUNT of Snow on Grnd (cm))
scs1 = snow_season_df[snow_season_df[stat1_col] > 0].groupby('snow_year').agg(count=(stat1_col, 'count'))
scs1_source = ColumnDataSource(data = dict(snow_year = scs1.index.values, count = scs1['count']))
scs1_fig = figure(title = 'Number of Snow Days of Snow on Ground (COUNT)', x_axis_label = 'Snow Year',
                  y_axis_label = 'Number of snow days', height=350, x_range = snow_years, toolbar_location=None)
scs1_fig.vbar(x='snow_year', top='count', width=0.5, color= 'LimeGreen', source = scs1_source)
scs1_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @count days</font>'))
scs1_fig.xgrid.grid_line_color = None
scs1_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels

grid1_figs = [sas1_fig, sms1_fig, sns1_fig, scs1_fig]
grid1 = gridplot(grid1_figs, ncols=2, width=400, height=350)
output_file('index1.html')
show(grid1)


             sum
snow_year       
SY2014     886.0
SY2015     662.0
SY2016     103.0
SY2017     278.0
SY2018     419.0
SY2019     487.0
SY2020     349.0
SY2021     315.0
SY2022     672.0
SY2023     500.0
SY2024      95.0
SY2025     836.0


Snow fall (attibute 'Total Snow (cm)') data analysis

In [34]:
''' Snow fall analysis * * * * * * * * * * * * * * * * * * * * * * * * * * '''
stat2_col = 'Total Snow (cm)'
# Figure 1: SAS - Snow Accumulation Season (SUM)
sas2 = snow_season_groups.agg(sum=(stat2_col, 'sum'))
sas2_source = ColumnDataSource(data = dict(snow_year = sas2.index.values, sum = sas2['sum']))
sas2_fig = figure(title = "Snow Accumulation of Season (SUM)", x_axis_label = 'Snow Year',
                  y_axis_label = 'Total Snow (cm)', height=350, x_range = snow_years, toolbar_location=None)
sas2_fig.vbar(x='snow_year', top='sum', width=0.5, color= 'LightSeaGreen', source = sas2_source)
sas2_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @sum (cm)</font>'))
sas2_fig.xgrid.grid_line_color = None
sas2_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels

# Figure 2: SMS - Daily Maximum Snow of Season (MAX)
sms2 = snow_season_groups.agg(max=(stat2_col, 'max'))
sms2_source = ColumnDataSource(data = dict(snow_year = sms2.index.values, max = sms2['max']))
sms2_fig = figure(title = "Daily Maximum Snow of Season (MAX)", x_axis_label = 'Snow Year',
                  y_axis_label = 'Maximum Snow (cm)', height=350, x_range = snow_years, toolbar_location=None)
sms2_fig.vbar(x='snow_year', top='max', width=0.5, color= 'Orange', source = sms2_source)
sms2_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @max (cm)</font>'))
sms2_fig.xgrid.grid_line_color = None
sms2_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels

# Figure 3: SES - Daily mean/average/norm snow of Season (MAX)
sns2 = snow_season_groups.agg(mean=(stat2_col, 'mean'))
sns2_source = ColumnDataSource(data = dict(snow_year = sns2.index.values, mean = sns2['mean']))
sns2_fig = figure(title = "Daily Mean Snow of Season (MEAN)", x_axis_label = 'Snow Year',
                  y_axis_label = 'Mean Snow (cm)', height=350, x_range = snow_years, toolbar_location=None)
sns2_fig.vbar(x='snow_year', top='mean', width=0.5, color= 'YellowGreen', source = sns2_source)
sns2_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @mean (cm)</font>'))
sns2_fig.xgrid.grid_line_color = None
sns2_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels

# Figure 4: SCS - Count Daily snow of Season (MAX)
scs2 = snow_season_df[snow_season_df[stat2_col] > 0].groupby('snow_year').agg(count=(stat2_col, 'count'))
scs2_source = ColumnDataSource(data = dict(snow_year = scs2.index.values, count = scs2['count']))
scs2_fig = figure(title = "Number of Snow Days of Season (COUNT)", x_axis_label = 'Snow Year',
                  y_axis_label = 'Number of snow days', height=350, x_range = snow_years, toolbar_location=None)
scs2_fig.vbar(x='snow_year', top='count', width=0.5, color= 'LawnGreen', source = scs2_source)
scs2_fig.add_tools(HoverTool(tooltips = '<font color=blue>@snow_year:</font><font color=red> @count days</font>'))
scs2_fig.xgrid.grid_line_color = None
scs2_fig.xaxis.major_label_orientation = math.pi / 4  # Rotate axis' labels


grid2_figs = [sas2_fig, sms2_fig, sns2_fig, scs2_fig]
grid2 = gridplot(grid2_figs, ncols=2, width=400, height=350)
output_file('index2.html')
show(grid2)


Based on above data analysis, SY2014 and SY2025 are the top of snow accumulation, and SY2025 has greatest daily maximum snow on ground. It requires to check daily snow on ground distribution.

Snow season SY2014 or SY2025 has extreme winter condition?

Following code is to generate daily snow on ground distribution to compare data from different years

In [35]:
from bokeh.palettes import Category20
from bokeh.models import CrosshairTool, CustomJSTickFormatter
from ipywidgets import interact, SelectMultiple, IntRangeSlider
from bokeh.io import push_notebook

# remove row with formatted datetime 'Feb 29' to make all years have same x-axis
snow_season_df = snow_season_df[snow_season_df['formatted_datetime'] != 'Feb 29']
# snow_season_df group by snow_year
snow_season_groups = snow_season_df.groupby('snow_year')
# snow season list (for lines chart) and snow year interaction list
snow_years = list(snow_season_groups.groups.keys())

# formatted datetime list (for x-axis) from Noverber 1st to April 30th
snow_season_dates = list(snow_season_df['formatted_datetime'].unique())

# Create a new figure
plot = figure(tooltips='$name - Snow on Ground on @x: @y2 cm',
           title='Daily snow on ground (cm)',
           x_axis_label='Days', y_axis_label='Snow Depth (cm)',
           x_range=snow_season_dates, width=800, height = 500)

# Define custom labels for the ticks (to solve long x-axis label issue)
xaxis_label_dict = {}
for i, val in enumerate(snow_season_dates):
    if (' 01' in val) or (' 15' in val)  or ('Apr 30' in val):
      xaxis_label_dict[val] = val
    else:
      xaxis_label_dict[val] = ""

# formate x-axis labels (keep 1st and 15th of each month)
plot.xaxis.formatter = CustomJSTickFormatter(
    args={'new_names': xaxis_label_dict},
    code='''
    return (tick in new_names) ? new_names[tick] : tick
    '''
)
plot.xaxis.major_label_orientation = math.pi / 4    # Rotate axis' labels

# add snow year lines or varea into plot
snow_season_lines = {}
for i, snow_year in enumerate(snow_years):
    snow_year_data = snow_season_groups.get_group(snow_year)
    snow_data = snow_year_data['Snow on Grnd (cm)']
    snow_season_line = plot.varea(x=snow_year_data['formatted_datetime'], y1=0, y2=snow_data,
                                  alpha=0.75, color=Category20[20][i], name=snow_year)
    snow_season_lines[snow_year] = snow_season_line
plot.add_tools(CrosshairTool())

# snow year chart interaction
for snow_season_line in snow_season_lines.values():
  snow_season_line.visible = False

handle = show(plot, notebook_handle=True)
snow_year_select = SelectMultiple(
    options=snow_season_groups.groups.keys(),
    rows=6,
    description='Show snow year:',
    style= {'description_width': 'initial'}
)
@interact(snow_seasons=snow_year_select)
def update(snow_seasons):
    for snow_season_name, snow_season_line in snow_season_lines.items():
        if snow_season_name in snow_seasons:
            snow_season_line.visible = True
        else:
            snow_season_line.visible = False
    initial_snow_seasons = ['SY2025']
    for snow_season in initial_snow_seasons:
      snow_season_lines[snow_season].visible = True
    push_notebook(handle=handle)

#output_file('index3.html')   #cannot export droplist to html
#save(plot)


interactive(children=(SelectMultiple(description='Show snow year:', options=('SY2014', 'SY2015', 'SY2016', 'SY…

Derive: Snow season SY2025 can be selected for snow flux model or snowdrift model.

---

Hourly datasets for winter wind analysis.
Main attributes for winter wind analysis:
- Temp (°C): temperature
- Wind Dir (10s deg): wind direction 10s in degree (example 31 (10s deg) = 310 degree
- Wind Spd (km/h): wind speed with unit km/h



In [36]:
import pandas as pd
import numpy as np
import plotly.express as px
''' init variables * * * * * * * * * * * * * * * * * * * * * * * * * * * * * '''
stationID = '51459'
sYear = 2013
eYear   = 2025
''' Step 1. read hourly csv files to data frame * * * * * * * * * * * * * * *'''
cols = ['Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)', 'Temp (°C)',
        'Precip. Amount (mm)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)']
df = pd.DataFrame()
for year in range(sYear, eYear+1):
  for month in range(1, 13):
    hourly_df = pd.read_csv(f'Station_{stationID}_{year}_{month}.csv',
                            usecols=cols, dtype={'Date/Time': str})
    df = pd.concat([df, hourly_df], ignore_index=True)


''' Step 2. Data Cleaning, Preparing * * * * * * * * * * * * * * * * * * * * '''
# Drop rows if  'Wind Dir (10s deg)' is empty (no wind or missing data)
df.dropna(axis = 0, subset = ['Wind Dir (10s deg)'], how = 'all', inplace = True)
# Drop rows if  'Temp (°C)' is positve (analysis only weather condition with
# temperature is below freezing)
df = df.drop(df.index[df['Temp (°C)'] > 0])
# Filter rows where 'month' is in period from November 1st to April 30th
snow_months = [1, 2, 3, 4, 11, 12]
df = df[df['Month'].isin(snow_months)]


# Convert wind direction (10s degree) to 16 cardinal wind directions or wind rose
#      0-11.25 = N, 11.25-33.75 = NNE,..., 326.25-348.75 = NNW, 348.75-360 = N
bins_dir = [0, 11.25, 33.75, 56.25, 78.75, 101.25, 123.75, 146.25, 168.75, 191.25,
            213.75, 236.25, 258.75, 281.25, 303.75, 326.25, 348.75, 360.00]
direction_names = ['N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSW','SW','WSW',
                   'W','WNW','NW','NNW', 'North']
df['wind_direction'] = pd.cut(df['Wind Dir (10s deg)']*10, bins_dir, labels=direction_names)
df['wind_direction'] = df['wind_direction'].replace('North', 'N')


# Create a classification wind speed to snowrose class (or wind speed class)
def snowrose_class_conditions(row):
    if (row['Wind Spd (km/h)'] > 0) & (row['Wind Spd (km/h)'] <= 10):
        val = '0-10km/h'
    elif (row['Wind Spd (km/h)'] > 10) & (row['Wind Spd (km/h)'] <= 20):
        val = '10-20km/h'
    elif (row['Wind Spd (km/h)'] > 20) & (row['Wind Spd (km/h)'] <= 30):
        val = '20-30km/h'
    elif (row['Wind Spd (km/h)'] > 30) & (row['Wind Spd (km/h)'] <= 40):
        val = '30-40km/h'
    elif (row['Wind Spd (km/h)'] > 40) & (row['Wind Spd (km/h)'] <= 50):
        val = '40-50km/h'
    elif (row['Wind Spd (km/h)'] > 50) & (row['Wind Spd (km/h)'] <= 60):
        val = '50-60km/h'
    elif (row['Wind Spd (km/h)'] > 60):
        val = '>60km/h'
    else:
        val = '0km/h'
    return val
# Apply snowrose_class_conditions function to each data row in the data frame
df['windrose_class']= df.apply(snowrose_class_conditions, axis=1)


''' Step 3. Wind direction and windrose statistics * * * * * * * * * * * * * * * * * * * * * * *'''
# Classify of wind direction and wind speed distribution
wind_stats = df.groupby(['wind_direction', 'windrose_class']).size().reset_index(name='percentage')
wind_stats['percentage'] = round(wind_stats['percentage']*100.0/df['wind_direction'].count(),2)

# Wind rose chart
fig = px.bar_polar(wind_stats, r="percentage", theta="wind_direction",
                   color="windrose_class", template="ygridoff",
                   color_discrete_sequence= px.colors.sequential.Plasma_r)
fig.update_layout(
    title=dict(text='Wind Speed Distribution at Toronto International Airport (closest station to hwy413)'),
    font_size=12,
    legend_font_size=12,
    polar_radialaxis_ticksuffix='%',
    polar_angularaxis_rotation=90,
    width = 800,
    height= 600
)
output_file('index.html')

fig.show()




/tmp/ipython-input-1270180440.py:37: FutureWarning:

The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.

/tmp/ipython-input-1270180440.py:65: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Derive:
Prevailing wind direction of GTA West Corridor area (Hwy413) is predominantly from the West (W) and West-Southwest (WSW) during snow season, also significant from North (N).
Snow fences work by being placed perpendicular to the prevailing winter wind direction.